# Implementación de Web Scraping en Diosa Bq
Este notebook contiene la implementación consolidada y funcional para extraer datos de **Diosa Bq Bodega Mayorista** utilizando BeautifulSoup, Selenium y Scrapy.

## Configuración del entorno Colab

In [1]:
!apt-get remove -y chromium-browser chromium-chromedriver
!wget -q -O - https://dl-ssl.google.com/linux/linux_signing_key.pub | apt-key add -
!sh -c 'echo "deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main" >> /etc/apt/sources.list.d/google-chrome.list'
!apt-get update
!apt-get install -y google-chrome-stable
!pip install selenium crochet scrapy bs4 requests

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
Package 'chromium-browser' is not installed, so not removed
Package 'chromium-chromedriver' is not installed, so not removed
0 upgraded, 0 newly installed, 0 to remove and 0 not upgraded.
OK
Get:1 http://dl.google.com/linux/chrome/deb stable InRelease [2,548 B]
Get:2 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Hit:5 http://archive.ubuntu.com/ubuntu noble InRelease
Get:6 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]
Get:7 http://dl.google.com/linux/chrome/deb stable/main amd64 Packages [1,414 B]
Get:8 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:9 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble

## 1. Implementación con BeautifulSoup

In [2]:
import requests
from bs4 import BeautifulSoup

url = 'https://www.diosabq.com/'
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36'
}

response = requests.get(url, headers=headers)
if response.status_code == 200:
    soup = BeautifulSoup(response.text, 'html.parser')

    tarjetas = soup.find_all('div', class_='card__content')

    print("--- RESULTADOS CON BEAUTIFULSOUP (Diosa Bq) ---")
    count = 0
    for tarjeta in tarjetas:
        if count >= 5: break

        titulo_elem = tarjeta.find(class_='card__heading')
        precio_elem = tarjeta.find(class_='price-item')

        if titulo_elem and precio_elem:
            titulo = titulo_elem.get_text(strip=True)
            precio = precio_elem.get_text(strip=True)
            if titulo and precio:
                print(f'Producto: {titulo} | Precio: {precio}')
                count += 1

    if count == 0:
        print("No se encontraron productos estructurados.")
else:
    print(f'Error de acceso HTTP: {response.status_code}')

--- RESULTADOS CON BEAUTIFULSOUP (Diosa Bq) ---
Producto: ESPEJO IPAD NEGRO GRANDE OG EIG-N-R | Precio: $17,000 COP
Producto: PESTAÑINA ALARGAMIENTO BOX OG M3353 | Precio: $20,000 COP
Producto: BROCHA PARA POLVO GUZEL GZ-101 | Precio: $12,000 COP
Producto: BROCHA PARA CORRECTOR GUZEL GZ-104 | Precio: $9,500 COP
Producto: BROCHA PARA BASE GUZEL GZ-102 | Precio: $15,000 COP


## 2. Implementación con Selenium

In [3]:
from selenium import webdriver
from selenium.webdriver.common.by import By
import time

options = webdriver.ChromeOptions()
options.add_argument('--headless=new')
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')
options.add_argument('--disable-gpu')
options.add_argument('--window-size=1920,1080')

try:
    driver = webdriver.Chrome(options=options)
    driver.get('https://www.diosabq.com/')

    driver.execute_script("window.scrollBy(0, 1000);")
    time.sleep(5)

    titulos = driver.find_elements(By.CSS_SELECTOR, '.card__heading')
    precios = driver.find_elements(By.CSS_SELECTOR, '.price-item')

    print("\n--- RESULTADOS CON SELENIUM (Diosa Bq) ---")
    count = 0
    for i in range(min(len(titulos), len(precios))):
        if count >= 5: break

        titulo = titulos[i].get_attribute('textContent').strip()
        precio = precios[i].get_attribute('textContent').strip()

        if titulo and precio and '$' in precio:
            titulo = " ".join(titulo.split())
            precio = " ".join(precio.split())
            print(f'Producto: {titulo} | Precio: {precio}')
            count += 1

    if count == 0:
        print("No se lograron aislar las tarjetas de producto.")

    driver.quit()
except Exception as e:
    print("Error en Selenium:", e)


--- RESULTADOS CON SELENIUM (Diosa Bq) ---
Producto: ESPEJO IPAD NEGRO GRANDE OG EIG-N-R | Precio: $17,000 COP
Producto: PESTAÑINA ALARGAMIENTO BOX OG M3353 | Precio: $17,000 COP
Producto: BROCHA PARA POLVO GUZEL GZ-101 | Precio: $20,000 COP
Producto: BROCHA PARA CORRECTOR GUZEL GZ-104 | Precio: $20,000 COP
Producto: BROCHA PARA BASE GUZEL GZ-102 | Precio: $12,000 COP


## 3. Implementación con Scrapy

In [4]:
import scrapy
from scrapy.crawler import CrawlerRunner
from crochet import setup, wait_for
import logging

setup()

from twisted.internet import reactor
installed_reactor = f"{reactor.__module__}.{reactor.__class__.__name__}"

class DiosaSpider(scrapy.Spider):
    name = 'diosa_spider'
    start_urls = ['https://www.diosabq.com/']

    custom_settings = {
        'LOG_LEVEL': logging.WARNING,
        'USER_AGENT': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)',
        'TWISTED_REACTOR': installed_reactor
    }

    def parse(self, response):
        print("\n--- RESULTADOS CON SCRAPY (Diosa Bq) ---")

        tarjetas = response.css('.card__content')

        count = 0
        for tarjeta in tarjetas:
            if count >= 5: break

            textos_titulo = tarjeta.css('.card__heading *::text').getall()
            titulo = "".join(textos_titulo).strip()

            textos_precio = tarjeta.css('.price-item *::text').getall()
            precio = "".join(textos_precio).strip()

            if titulo and precio and '$' in precio:
                titulo = " ".join(titulo.split())
                precio = " ".join(precio.split())

                print(f'Producto: {titulo} | Precio: {precio}')
                count += 1

        if count == 0:
            print("No se extrajeron productos con Scrapy.")

@wait_for(timeout=30.0)
def run_spider():
    runner = CrawlerRunner()
    deferred = runner.crawl(DiosaSpider)
    return deferred

run_spider()
print("Scraping completado mediante Scrapy.")


--- RESULTADOS CON SCRAPY (Diosa Bq) ---
Producto: ESPEJO IPAD NEGRO GRANDE OG EIG-N-R | Precio: $17,000 COP $17,000 COP / por
Producto: PESTAÑINA ALARGAMIENTO BOX OG M3353 | Precio: $20,000 COP $20,000 COP / por
Producto: BROCHA PARA POLVO GUZEL GZ-101 | Precio: $12,000 COP $12,000 COP / por
Producto: BROCHA PARA CORRECTOR GUZEL GZ-104 | Precio: $9,500 COP $9,500 COP / por
Producto: BROCHA PARA BASE GUZEL GZ-102 | Precio: $15,000 COP $15,000 COP / por
Scraping completado mediante Scrapy.
